# Смертность в Москве: до пандемии COVID-19 и во время неё

Проверяем, отличается ли среднемесячное число смертей в двух периодах.

- **H0:** среднемесячная смертность до пандемии и во время пандемии одинакова.
- **H1:** среднемесячная смертность различается.

Уровень значимости α = 0,05.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

df_1 = pd.read_csv('moscow_data.csv', sep=';')
df_1.columns = df_1.columns.str.strip()
df_1.head()

## Подготовка данных

Переводим таблицу «год × месяц» в один временной ряд по месяцам.

In [ ]:
months = list(df_1.columns[1:])
month_num = {m: i + 1 for i, m in enumerate(months)}

long = df_1.melt(id_vars='Год', var_name='Месяц', value_name='deaths').dropna()
long['date'] = pd.to_datetime(dict(year=long['Год'], month=long['Месяц'].map(month_num), day=1))
deaths = long.sort_values('date').set_index('date')['deaths'].astype(float)

print(f'{len(deaths)} месяцев: {deaths.index.min():%Y-%m} – {deaths.index.max():%Y-%m}')

## Периоды

- **До пандемии:** январь 2006 – март 2020.
- **Пандемия:** апрель 2020 – март 2022 (от первого месяца избыточной смертности до её исчезновения).

In [ ]:
pre = deaths[:'2020-03']
covid = deaths['2020-04':'2022-03']

summary = pd.DataFrame({'До пандемии': pre.describe(), 'Пандемия': covid.describe()}).round(1)
diff = covid.mean() - pre.mean()
print(f'Разница средних: {diff:+.0f} смертей в месяц ({diff / pre.mean():+.1%})')
summary

## Проверка предпосылок

Нормальность (Шапиро–Уилк) и равенство дисперсий (Левене) определяют выбор теста.

In [ ]:
print(f'Шапиро–Уилк, до пандемии: p = {stats.shapiro(pre).pvalue:.3g}')
print(f'Шапиро–Уилк, пандемия:    p = {stats.shapiro(covid).pvalue:.3g}')
print(f'Левене (равенство дисперсий): p = {stats.levene(pre, covid).pvalue:.3g}')

Допандемийный ряд не нормален, дисперсии не равны, поэтому t-тест Стьюдента не подходит. Основной тест — Манна–Уитни, дополнительно t-тест Уэлча и перестановочный тест.

In [ ]:
welch = stats.ttest_ind(covid, pre, equal_var=False)
ci = welch.confidence_interval(0.95)
mwu = stats.mannwhitneyu(covid, pre, alternative='two-sided')
perm = stats.permutation_test((covid.values, pre.values), lambda x, y: x.mean() - y.mean(),
                              n_resamples=20000, random_state=0)

# размеры эффекта
n1, n2 = len(pre), len(covid)
sd_pooled = np.sqrt(((n1 - 1) * pre.var() + (n2 - 1) * covid.var()) / (n1 + n2 - 2))
cohen_d = diff / sd_pooled
rank_biserial = 2 * mwu.statistic / (n1 * n2) - 1

print(f'Манна–Уитни:     U = {mwu.statistic:.0f}, p = {mwu.pvalue:.3g}')
print(f't-тест Уэлча:    t = {welch.statistic:.2f}, df = {welch.df:.1f}, p = {welch.pvalue:.3g}')
print(f'Перестановочный: p = {perm.pvalue:.3g}')
print(f'95% ДИ разницы средних: [{ci.low:.0f}; {ci.high:.0f}]')
print(f'd Коэна = {cohen_d:.2f}, ранг-бисериальная корреляция = {rank_biserial:.2f}')

## Учёт сезонности

Смертность зависит от времени года, поэтому сравниваем одинаковые календарные месяцы: среднее за январи до пандемии с январями пандемии и т. д. (12 пар).

In [ ]:
by_month = pd.DataFrame({
    'До пандемии': pre.groupby(pre.index.month).mean(),
    'Пандемия': covid.groupby(covid.index.month).mean(),
})
by_month.index = months
by_month['Разница, %'] = (by_month['Пандемия'] / by_month['До пандемии'] - 1) * 100

paired_t = stats.ttest_rel(by_month['Пандемия'], by_month['До пандемии'])
wilcoxon = stats.wilcoxon(by_month['Пандемия'], by_month['До пандемии'])
print(f'Парный t-тест: t = {paired_t.statistic:.2f}, p = {paired_t.pvalue:.3g}')
print(f'Уилкоксон:     W = {wilcoxon.statistic:.0f}, p = {wilcoxon.pvalue:.3g}')
by_month.round(1)

## Устойчивость результата

Те же тесты при других границах периодов.

In [ ]:
def compare(pre, other):
    d = other.mean() - pre.mean()
    return {
        'n до': len(pre), 'n после': len(other),
        'Среднее до': pre.mean(), 'Среднее после': other.mean(),
        'Разница, %': d / pre.mean() * 100,
        'p Манна–Уитни': stats.mannwhitneyu(other, pre).pvalue,
        'p Уэлча': stats.ttest_ind(other, pre, equal_var=False).pvalue,
    }

heat_2010 = pd.to_datetime(['2010-07-01', '2010-08-01'])
variants = {
    'Основной: 2006-01..2020-03 и 2020-04..2022-03': (pre, covid),
    'Без жары июля–августа 2010': (pre.drop(heat_2010), covid),
    'База только 2015-01..2020-03': (deaths['2015-01':'2020-03'], covid),
    'Пандемия до 2023-04 (отмена режима ЧС ВОЗ)': (pre, deaths['2020-04':'2023-04']),
    'После пандемии: 2022-04..2024-06': (pre, deaths['2022-04':]),
}
robust = pd.DataFrame({name: compare(a, b) for name, (a, b) in variants.items()}).T
for col in ['p Манна–Уитни', 'p Уэлча']:
    robust[col] = robust[col].map('{:.2g}'.format)
robust['Разница, %'] = robust['Разница, %'].map('{:+.1f}'.format)
robust.round(0)

## Вывод

H0 отвергается: среднемесячная смертность в Москве во время пандемии (13 814) статистически значимо выше, чем до неё (10 091), — на 37%, p < 0,001 по всем тестам, d Коэна = 3,6. Превышение наблюдается во все 12 календарных месяцев и не зависит от выбора границ периодов.

Ограничение: месячные значения автокоррелированы, поэтому p-value несколько занижены; при таком размере эффекта на вывод это не влияет.